In [2]:
import os
from dotenv import load_dotenv
from langsmith import Client
opanai_key = os.getenv("OPENAI_PI_KEY")
langsmith_key = os.getenv("LANGSMITH_API_KEY")
langsmith_tracing = True
load_dotenv(override=True)

True

In [3]:
client = Client()
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id= dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }        
    ]
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

## Define Metrics (LLM as a Judge) ##

In [4]:
import openai
from langsmith import wrappers

openai_client =wrappers.wrap_openai(openai.OpenAI())
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict)-> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role":"system", "content":eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content
    
    return response == "CORRECT"
    

In [5]:
# Concisions : Checks whether the actual output is less than 2x the length of expected result.

def concision(outputs: dict, reference_outputs: dict)-> bool:
    return int(len(outputs["response"])) < 2 * len(reference_outputs["answer"])

In [10]:
## Running the Evaluation ##
default_instructions = "Respond to the users question in a short, concise manner(one short sentence)."
def my_app(question: str, instructions: str = default_instructions)-> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system", "content": instructions},
            {"role":"user", "content":question}
        ]
    ).choices[0].message.content
    return response

In [13]:
# calling my_app for every datapoints
def ls_target(inputs: str)-> dict:
    return {"response": my_app(inputs["question"])}

In [14]:
## Run for evaluation
experimental_results= client.evaluate(
    ls_target, # AI System
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix= "openai-4o-mini"
)

experimental_results

View the evaluation results for experiment: 'openai-4o-mini-a97e2a4d' at:
https://smith.langchain.com/o/ded64c70-97b2-4828-9487-f10ed0ed0c43/datasets/224c90e8-6402-44c9-afa8-5487c04ffac4/compare?selectedSessions=1f4edbe3-267d-4b59-96de-65b2bdc71ce0




5it [00:12,  2.55s/it]


<ExperimentResults openai-4o-mini-a97e2a4d>

## Evaluation For RAG ##

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfminer

# List of Urls to load documents from

urls = [
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf",
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/b.pdf",
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/c.pdf",
]

#load documents from Urls
docs = [PyPDFLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

#split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

#Add the document chunks to the "vector store" using the OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embedding = OpenAIEmbeddings()
)

#with using langchain we can turn vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

In [9]:
retriever.invoke("What is agents?")

[Document(id='63da9151-90c8-482e-bef2-1caa2a17cc25', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'LLM Powered Autonomous Agents', 'source': 'https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="LLM Powered Autonomous AgentsDate: June 23, 2023  |  Author: Lilian Weng\nSource: Lil'Log (https://lilianweng.github.io/posts/2023-06-23-agent/)\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several\nproof-of-concepts  demos,  such  as  AutoGPT,  GPT-Engineer  and  BabyAGI,  serve  as  inspiring\nexamples. The potentiality of LLM extends beyond generating well-written copies, stories, essays\nand programs; it can be framed as a powerful general problem solver .\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent's brain, complemented\nby several key components:\nPlanning:\nSubgoal a

In [10]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F4417DEE10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F4417DCEF0>, root_client=<openai.OpenAI object at 0x000001F4417DCC80>, root_async_client=<openai.AsyncOpenAI object at 0x000001F4417DF050>, model_name='gpt-4o-mini', model_kwargs={}, 

In [11]:
from langsmith import traceable

@traceable
def rag_bot(question: str)-> dict:
    ## Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use two sentences maximum and keep the answer concise.
                        Documents: {docs_string}"""
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    return {"answer": ai_msg.content, "documents": docs}                        

In [12]:
rag_bot("What is agents")

{'answer': 'Agents, in the context of LLM-powered autonomous systems, are entities that function using a large language model as their core controller, capable of planning, reflecting, learning, and using tools to perform tasks effectively. They are designed to solve complex problems by breaking down tasks into manageable subgoals and leveraging external resources.',
 'documents': [Document(id='63da9151-90c8-482e-bef2-1caa2a17cc25', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'LLM Powered Autonomous Agents', 'source': 'https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="LLM Powered Autonomous AgentsDate: June 23, 2023  |  Author: Lilian Weng\nSource: Lil'Log (https://lilianweng.github.io/posts/2023-06-23-agent/)\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several\nproof-of-concepts  demos,  such  as  AutoGPT, 

### Dataset for RAG Evaluation ###

In [15]:
from langsmith import Client
client = Client()

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."
        },
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {
            "answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."
        },
    },
]


dataset_name = 'Evalation for RAG'
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

{'example_ids': ['68817aa5-fda9-4b67-8f42-9fe67e3f2f7c',
  '26319287-4219-430d-bbcf-4172c538af43',
  'eeb5bbeb-9ce5-4c77-ae7f-433419f963f6'],
 'count': 3,
 'as_of': '2026-06-08T15:16:09.354425787Z'}